In [7]:
# Importar las bibliotecas necesarias
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Cargar el dataset
dataset = pd.read_csv('student_scores.csv')

# Revisar las columnas del dataset
print(dataset.head())

# Preparar el dataset para la regresión
# Definir las características (X) y la variable objetivo (y)
X = dataset[['Hours']]  # Característica
y = dataset['Scores']   # Variable objetivo

# Dividir el dataset en entrenamiento y prueba (25% para el conjunto de prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Convertir los datos en tensores para PyTorch
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Definir la red neuronal simplificada
class FeedForwardNN(nn.Module):
    def __init__(self):
        super(FeedForwardNN, self).__init__()
        self.fc1 = nn.Linear(X_train_tensor.shape[1], 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Crear el modelo de red neuronal
model = FeedForwardNN()

# Definir la función de pérdida (MSE) y el optimizador (Adam)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0) 

# Entrenamiento del modelo
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Evaluación del modelo
model.eval()
y_pred = model(X_test_tensor).detach().numpy()

# Calcular RMSE manualmente (sin usar mean_squared_error)
mse = np.mean((y_test.values - y_pred.flatten()) ** 2)  # Mean Squared Error manual
rmse = np.sqrt(mse)  # Root Mean Squared Error (RMSE)

# Calcular las métricas de evaluación
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'Red Neuronal - R2: {r2:.4f}')
print(f'Red Neuronal - MAE: {mae:.4f}')
print(f'Red Neuronal - RMSE: {rmse:.4f}')

   Hours  Scores
0    2.5      21
1    5.1      47
2    3.2      27
3    8.5      75
4    3.5      30
Epoch [10/200], Loss: 3279.1052
Epoch [20/200], Loss: 3118.7759
Epoch [30/200], Loss: 2923.8276
Epoch [40/200], Loss: 2687.3171
Epoch [50/200], Loss: 2400.0283
Epoch [60/200], Loss: 2065.2275
Epoch [70/200], Loss: 1694.6630
Epoch [80/200], Loss: 1308.1184
Epoch [90/200], Loss: 931.8010
Epoch [100/200], Loss: 597.9827
Epoch [110/200], Loss: 336.5160
Epoch [120/200], Loss: 163.1468
Epoch [130/200], Loss: 72.0720
Epoch [140/200], Loss: 38.0654
Epoch [150/200], Loss: 31.0182
Epoch [160/200], Loss: 31.0582
Epoch [170/200], Loss: 31.2306
Epoch [180/200], Loss: 30.9744
Epoch [190/200], Loss: 30.7596
Epoch [200/200], Loss: 30.6771
Red Neuronal - R2: 0.9527
Red Neuronal - MAE: 4.4348
Red Neuronal - RMSE: 4.9891
